# 基于 LayoutXLM 的语义实体识别微调

在本教程中，我们将使用**中文版版多语言 XFUND 数据集**对 LayoutLMv2ForTokenClassification 进行微调。该任务的目标是为扫描文档中的单词添加适当的语义标签。我们将基于 [microsoft/layoutxlm-base](https://huggingface.co/microsoft/layoutxlm-base) 的预训练权重开始，该模型已在七种语言（中文、日文、西班牙文、法文、意大利文、德文、葡萄牙文）的数百万份多语言文档上进行预训练。 

此任务被视为 NER 问题（序列标记）。但是，与 BERT 相比，LayoutLMv2 在将token编码为向量时还包含有关token的视觉和布局信息。这使得 LayoutLMv2 模型对于文档理解任务非常强大。 

LayoutLMv2 本身是 LayoutLM 的升级版。LayoutLMv2 的主要新颖之处在于它还预先训练视觉嵌入，而原始 LayoutLM 仅在微调期间添加视觉嵌入。LayoutXLM 等效于 LayoutLMv2，唯一的区别是预训练的权重。 

- [LayoutLMv2 论文](https://arxiv.org/abs/2012.14740) - 2020 年 12 月 
- [LayoutXLM 论文](https://arxiv.org/abs/2104.08836) - 2021 年 4 月 
- Microsoft UniLM 项目: [github.com/microsoft/unilm](https://github.com/microsoft/unilm) 



在环境搭建部分，使用了AI gallery社区中相关mindnlp项目搭建mindnlp环境的代码。  
## 环境配置

配置python3.9环境

In [ ]:
%%capture captured_output
!/home/ma-user/anaconda3/bin/conda create -n python-3.9.0 python=3.9.0 -y --override-channels --channel https://mirrors.tuna.tsinghua.edu.cn/anaconda/pkgs/main
!/home/ma-user/anaconda3/envs/python-3.9.0/bin/pip install ipykernel

In [ ]:
import json
import os

data = {
   "display_name": "python-3.9.0",
   "env": {
      "PATH": "/home/ma-user/anaconda3/envs/python-3.9.0/bin:/home/ma-user/anaconda3/envs/python-3.7.10/bin:/modelarts/authoring/notebook-conda/bin:/opt/conda/bin:/usr/local/nvidia/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/home/ma-user/modelarts/ma-cli/bin:/home/ma-user/modelarts/ma-cli/bin"
   },
   "language": "python",
   "argv": [
      "/home/ma-user/anaconda3/envs/python-3.9.0/bin/python",
      "-m",
      "ipykernel",
      "-f",
      "{connection_file}"
   ]
}

if not os.path.exists("/home/ma-user/anaconda3/share/jupyter/kernels/python-3.9.0/"):
    os.mkdir("/home/ma-user/anaconda3/share/jupyter/kernels/python-3.9.0/")

with open('/home/ma-user/anaconda3/share/jupyter/kernels/python-3.9.0/kernel.json', 'w') as f:
    json.dump(data, f, indent=4)

注：以上代码执行完成后，稍等片刻，或刷新页面，点击右上角（或左上角）kernel选择python-3.9.0 change-kernel

下面安装相关依赖

In [ ]:
!pip install mindspore
!pip install mindnlp==0.4
!pip install setuptools_scm
!pip install seqeval
!pip uninstall mindformers -y

检查mindspore版本是否正确,在这里我们选用目标设备为Ascend,也可以根据自己的实际平台需求改为GPU或CPU

In [ ]:
import mindspore
import mindnlp
mindspore.set_context(device_target='Ascend')
mindspore.run_check()

下载FUNSD数据集的中文部分,假如网络太慢可以自己从本地上传到xfund_data目录下

In [ ]:
import os
import zipfile
import requests

urls = [
    "https://github.com/doc-analysis/XFUND/releases/download/v1.0/zh.train.json",
    "https://github.com/doc-analysis/XFUND/releases/download/v1.0/zh.val.json",
    "https://github.com/doc-analysis/XFUND/releases/download/v1.0/zh.val.zip",
    "https://github.com/doc-analysis/XFUND/releases/download/v1.0/zh.train.zip"
]

data_dir = "xfund_data"
if not os.path.exists(data_dir):
    os.makedirs(data_dir)

for url in urls:
    filename = os.path.join(data_dir, os.path.basename(url))
    if not os.path.exists(filename):
        response = requests.get(url)
        with open(filename, 'wb') as f:
            f.write(response.content)
    if filename.endswith('.zip'):
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall(data_dir)

# 准备数据
下面我们实现自定义数据集类来加载XFUN数据集。

注意：  
这个数据集已经是模型所期望的格式，因此不需要更复杂的数据处理。

由于微软的作者已经为我们准备好了数据，标注了文本框和标签，因此我们唯一需要编写的是一个数据加载器，使用它来将示例数据进行预处理处理并作为训练数据集和测试数据集。  
我们可以利用分词器将汉字转为令牌令牌级的input_ids，文本框和NER标签也需要转换为标准化的bbox、labels。 
此外，也需要将图像转为模型所期望的形状，在这里是进行像素标准化并缩小到长和宽为224的RGB图像。  

首先我们需要创建一个id2label和label2id映射，将标签映射为id，这在训练后进行推理时非常有用。


## 定义数据整理器
下面，我们定义这个数据加载器。对于文本输入，我们使用bert分词器进行处理，对于文本框，缩放到0到1000，对于图像，缩放到224x224并标准化。
 
首先加载数据并定义标签映射。 

In [ ]:
import json
import os
data_dir = "xfund_data"
def load_json_data(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return data

train_data = load_json_data(os.path.join(data_dir, 'zh.train.json'))
val_data = load_json_data(os.path.join(data_dir, 'zh.val.json'))

# 定义标签映射
id2label = {0: 'O', 1: 'B-QUESTION', 2: 'B-ANSWER', 3: 'B-HEADER', 4: 'I-ANSWER', 5: 'I-QUESTION', 6: 'I-HEADER'}
label2id = {v: k for k, v in id2label.items()}

然后我们来定义一个数据加载类对数据进行处理，使其适合模型输入。  
使用分词器得到token，对文本框进行归一化，并将输入进行截取和填充。  
随后分别实例化一个训练数据集和测试数据集。

In [ ]:
import mindspore
from mindnlp.transformers import BertTokenizer
import numpy as np
from PIL import Image
tokenizer = BertTokenizer.from_pretrained("bert-base-chinese")

class XFUNSDataset():
    def __init__(self, data, tokenizer, max_length, target_size, label2id,train=True,data_dir="./xfund_data"):
        self.data = data
        self.tokenizer = tokenizer
        self.max_seq_length = max_length
        self.target_size = target_size
        self.pad_token_box = [0, 0, 0, 0]
        self.train = train
        self.data_dir = data_dir
        self.label2id = label2id
        self.input_ids = []
        self.normalized_word_boxes = []
        self.attention_mask = []
        self.image_tensor= []
        self.labels = []
        
        
        
        for idx in range(len(self.data['documents'])):
            item = self.data['documents'][idx]
            base_path = self.data_dir
            img_path = os.path.join(base_path, item['img']['fname'])
            print(f"尝试加载图像路径：{img_path}")  # 定位具体出错文件
            assert os.path.exists(img_path), f"文件{img_path}不存在"
            original_image = Image.open(img_path).convert("RGB")
            resized_image = original_image.resize((self.target_size, self.target_size))

            unnormalized_word_boxes = []
            labels = []
            input_ids = []
            attention_mask = []

            for block in item['document']:
                flag = True
                label = block['label']
                for annotation in block['words']:
                    if annotation['text'] == '':
                        continue
                    temp = self.tokenizer(annotation['text'], add_special_tokens=False)
                    temp_ids = temp["input_ids"]
                    unnormalized_word_boxes = unnormalized_word_boxes + [annotation['box'] for i in range(len(temp_ids))]
                    input_ids = input_ids + temp_ids

                    for i in range(len(temp_ids)):
                        if label!="other":
                            if flag:
                                temp_label = 'B-' + label.upper()
                                flag = False
                            else:
                                temp_label = 'I-' + label.upper()
                        else:
                            temp_label = 'O'
                        labels.append(self.label2id[temp_label])

            width, height = original_image.size
            normalized_word_boxes = [self.normalize_box(bbox, width, height) for bbox in unnormalized_word_boxes]
            assert len(input_ids) == len(normalized_word_boxes)

            # max length

            special_tokens_count = 2
            if len(input_ids) > self.max_seq_length - special_tokens_count:
                input_ids = input_ids[:self.max_seq_length - special_tokens_count]
                normalized_word_boxes = normalized_word_boxes[:self.max_seq_length - special_tokens_count]
                labels = labels[:self.max_seq_length - special_tokens_count]

            normalized_word_boxes = [[0,0,0,0]] + normalized_word_boxes + [[1000,1000,1000,1000]]
            labels = [-100] + labels + [-100]
            input_ids = [tokenizer.cls_token_id] + input_ids + [tokenizer.sep_token_id]

            # pad
            tokken_length = len(input_ids)
            padding_length = self.max_seq_length-tokken_length
            labels = labels + [-100]*padding_length
            input_ids = input_ids + [self.tokenizer.pad_token_id]*padding_length
            normalized_word_boxes = normalized_word_boxes + [[0,0,0,0]]*padding_length
            attention_mask = [0] + [1]*(tokken_length-2) + [0]*(padding_length+1)

            #tensor

            mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
            std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
            image_np = (np.array(resized_image).astype(np.float32) / 255.0 - mean) / std
            image_np = image_np.transpose(2, 0, 1)  # HWC to CHW
            
            input_ids = np.array(input_ids, dtype=np.int64)
            labels = np.array(labels, dtype=np.int32)
            normalized_word_boxes = np.array(normalized_word_boxes, dtype=np.int64)
            attention_mask = np.array(attention_mask, dtype=np.int64)
            
            self.input_ids.append(input_ids)
            self.normalized_word_boxes.append(normalized_word_boxes)
            self.attention_mask.append(attention_mask)
            self.image_tensor.append(image_np)
            self.labels.append(labels)

    def __len__(self):
        return len(self.data['documents'])
    def normalize_box(self,box, width, height):
        return [
            int(1000 * (box[0] / width)),
            int(1000 * (box[1] / height)),
            int(1000 * (box[2] / width)),
            int(1000 * (box[3] / height)),
        ]

    def __getitem__(self, idx):
        return(
            mindspore.Tensor(self.input_ids[idx], dtype=mindspore.int64),      # input_ids
            mindspore.Tensor(self.normalized_word_boxes[idx], dtype=mindspore.int64),
            mindspore.Tensor(self.attention_mask[idx], dtype=mindspore.int64), # attention_mask
            mindspore.Tensor(self.image_tensor[idx], dtype=mindspore.float32), # image
            mindspore.Tensor(self.labels[idx], dtype=mindspore.int32)          # labels
        )
        # return self.input_ids[idx],self.normalized_word_boxes[idx],self.attention_mask[idx],self.image_tensor[idx],self.labels[idx]
    
train_data_loader = XFUNSDataset(train_data,tokenizer,max_length=512,target_size=224,label2id=label2id)
val_data_loader = XFUNSDataset(val_data,tokenizer,max_length=512,target_size=224,label2id=label2id)



## 创建Dataset
接下来，我们可以创建Dataset 并在mindspore中训练模型。

我们只需要使用上面创建的可以迭代的数据加载类，使用GeneratorDataset就可以得到一个可以使用的数据集

下面，我们创建两个数据集，一个是训练数据集，一个是测试数据集，并且批量读取。

In [ ]:
from mindspore.dataset import GeneratorDataset

train_ms_dataset = GeneratorDataset(train_data_loader, column_names=["input_ids", "bbox", "attention_mask","img","ner_tags"])
val_ms_dataset = GeneratorDataset(val_data_loader, column_names=["input_ids", "bbox", "attention_mask","img","ner_tags"])

# 批量处理
train_ms_dataset = train_ms_dataset.batch(4)
val_ms_dataset = val_ms_dataset.batch(4)

我们随机取一个批次的数据来确认下数据处理正确  
先打印下数据的形状


In [ ]:
from PIL import Image, ImageDraw


batch = next(train_ms_dataset.create_dict_iterator())

for k, v in batch.items():
    print(k, v.shape,v.dtype)

其次我们输出文本输入以及对于的标签

In [ ]:
# 解码输入的标记
decoded_text = tokenizer.decode(batch['input_ids'][0])
print("解码后的输入文本:", decoded_text)

# 打印前 30 个标记及其对应的标签
for id, label in zip(batch['input_ids'][0][1:10], batch['ner_tags'][0][1:10]):
    decoded_token = tokenizer.decode(id)
    print(decoded_token, id2label[label.asnumpy().item()])

我们再来确认下对图像数据的处理是否正确。

In [ ]:
import numpy as np
from PIL import Image

image_to_verify = batch['img'][0].numpy()
image_to_verify = np.moveaxis(image_to_verify, source=0, destination=-1)
mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
image_to_verify = (image_to_verify * std) + mean
image_to_verify = (image_to_verify * 255).astype(np.uint8)

image_to_verify = Image.fromarray(image_to_verify)
image_to_verify

最后确认一下文本框的处理是否正确

In [ ]:
from PIL import ImageDraw

def unnormalize_box(bbox, width, height):
     return [
         width * (bbox[0] / 1000),
         height * (bbox[1] / 1000),
         width * (bbox[2] / 1000),
         height * (bbox[3] / 1000),
     ]

draw = ImageDraw.Draw(image_to_verify)
for bbox in batch['bbox'][0]:
    
    draw.rectangle(unnormalize_box(bbox.asnumpy(), width=224, height=224), outline='red', width=1)

image_to_verify

## 定义模型
我们首先定义模型。  
作为分类使用的该模型将基础 Transformer 的权重与预先训练的权重实例化，并在顶部添加一个随机初始化的 head（标签分类器 head）。  
该分类器将与预训练好的基础模型一起进行微调。

In [ ]:
from mindnlp.transformers import LayoutLMv2ForTokenClassification

model = LayoutLMv2ForTokenClassification.from_pretrained('microsoft/layoutxlm-base',
                                                         id2label=id2label,
                                                         label2id=label2id,
                                                        ignore_mismatched_sizes=True)

In [ ]:
import mindspore as ms

# 获取中间层权重
dense_weights = model.layoutlmv2.encoder.layer[0].intermediate.dense.weight.data
print("权重矩阵维度:", dense_weights.shape)  # 应为(3072, 768)

# 示例输出：
# 权重矩阵维度: (3072, 768)


## 训练模型
我们使用AdamW对模型进行训练，学习率为5e-5。

In [ ]:
import mindspore.nn as nn
import mindspore.ops as ops
import mindspore.context as context
from tqdm import tqdm
from mindnlp.core.optim import AdamW

from mindspore.experimental import optim
optimizer = AdamW(model.trainable_params(), lr =5e-5)


loss_fn = nn.CrossEntropyLoss()
def forward_fn(input_ids, bbox, image, labels,attention_mask):
    outputs = model(
        input_ids=input_ids,
        bbox=bbox,
        image=image,
        attention_mask=attention_mask,
        labels = labels
    )
    
    return outputs.loss


grad_fn = mindspore.value_and_grad(forward_fn, None, model.trainable_params(),has_aux=False)
# 训练循环
num_epochs = 10
model.set_train()
for epoch in range(num_epochs):
    all_loss = 0.0
    for batch in tqdm(train_ms_dataset.create_dict_iterator(),total=train_ms_dataset.get_dataset_size()):
        input_ids = batch['input_ids']
        bbox = batch['bbox']
        image = batch['img']
        labels = batch['ner_tags']
        attention_mask = batch["attention_mask"]

        loss, grads = grad_fn(input_ids,bbox,image,labels,attention_mask)
        optimizer.step(grads)
        
        all_loss = all_loss + loss
    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {all_loss/train_ms_dataset.get_dataset_size()}")
    # mindspore.save_checkpoint(model, f"model_epoch{epoch}.ckpt")

微调后的模型我们可以先保存

In [ ]:
model.save_pretrained("./Checkpoints20250227")

## 评估模型  

使用seqeval.metrics对模型的预测进行评估。

In [ ]:
import numpy as np
from tqdm import tqdm
from seqeval.metrics import (
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)

eval_loss = 0.0
nb_eval_steps = 0
preds = []  # 存储有效预测
true_labels = []  # 存储有效标签
model.set_train(False)  # 禁用训练模式

for batch in tqdm(val_ms_dataset.create_dict_iterator(), total=val_ms_dataset.get_dataset_size()):
    
    input_ids = batch['input_ids']
    bbox = batch['bbox']
    image = batch['img']
    labels = batch['ner_tags']
    attention_mask = batch["attention_mask"]
    
    
    outputs = model(
        input_ids=input_ids,
        bbox=bbox,
        image=image,
        labels=labels,
        attention_mask=attention_mask
    )
    
    # 损失
    tmp_eval_loss = outputs.loss
    eval_loss += tmp_eval_loss.item()
    nb_eval_steps += 1

    # 获取预测结果
    logits = outputs.logits.asnumpy()  # shape: (batch_size, seq_len, num_labels)
    batch_pred_ids = np.argmax(logits, axis=-1)  # shape: (batch_size, seq_len)
    batch_labels = labels.asnumpy()  # shape: (batch_size, seq_len)
    batch_attention = attention_mask.asnumpy().astype(bool)  # shape: (batch_size, seq_len)
    
    for i in range(batch_pred_ids.shape[0]):
        valid_mask = batch_attention[i]
        
        valid_pred = batch_pred_ids[i][valid_mask].tolist()
        valid_label = batch_labels[i][valid_mask].tolist()
        preds.append(valid_pred)
        true_labels.append(valid_label)

# 计算平均损失
eval_loss = eval_loss / nb_eval_steps

# 转换为标签
preds_text = [[id2label[p] for p in pred_seq] for pred_seq in preds]
labels_text = [[id2label[l] for l in label_seq] for label_seq in true_labels]

# 生成评估
results = {
    "loss": eval_loss,
    "precision": precision_score(labels_text, preds_text),
    "recall": recall_score(labels_text, preds_text),
    "f1": f1_score(labels_text, preds_text),
    "detail_report": classification_report(labels_text, preds_text)
}

print(results)

## 测试模型
接下来测试训练后的模型，我们可以先取一个文档进行测试。  
加载文档的图片，获取文档的数据。  

In [ ]:
import numpy as np
import mindspore.ops as ops

item = val_data_loader.data['documents'][2]
base_path = "./xfund_data"
img_path = os.path.join(base_path, item['img']['fname'])
print(f"尝试加载图像路径：{img_path}")  # 定位具体出错文件
assert os.path.exists(img_path), f"文件{img_path}不存在"
original_image = Image.open(img_path).convert("RGB")


input_ids,bbox,attention_mask,image,labels = val_data_loader[2]
input_ids = ops.expand_dims(input_ids, axis=0)        # -> [1, seq_len]
print(input_ids.shape)
bbox = ops.expand_dims(bbox, axis=0) 
print(bbox.shape)
image = ops.expand_dims(image, axis=0)
print(image.shape)
labels = ops.expand_dims(labels, axis=0)
print(labels.shape)
attention_mask = ops.expand_dims(attention_mask, axis=0)  
print(attention_mask.shape)

outputs = model(input_ids=input_ids, bbox=bbox, image=image, labels=labels,attention_mask=attention_mask)
pred_ids = np.argmax(outputs.logits.asnumpy(), axis=-1)[0]
original_image


接下来对模型进行验证  
首先将数据输入模型得到预测后的标签  
画出文本框和对应的标签，为了方便与正确标签进行对比，不同标签使用不同的颜色显示。
可以看到分类结果基本正确

In [ ]:
from PIL import Image, ImageDraw, ImageFont


def unnormalize_box(bbox, width, height):
     return [
         width * (bbox[0] / 1000),
         height * (bbox[1] / 1000),
         width * (bbox[2] / 1000),
         height * (bbox[3] / 1000),
     ]




draw = ImageDraw.Draw(original_image)
font = ImageFont.load_default()
width, height = original_image.size
def iob_to_label(label):
    if label=="O": return "o"
    return label[2:].lower()

label2color = {'question':'blue', 'answer':'green', 'header':'orange', 'o':'violet'}

for prediction,true_id,box in zip(pred_ids, labels[0].asnumpy(), bbox[0].asnumpy()):
    if true_id==-100: continue
    unnormal_box = unnormalize_box(box, width, height)
    predicted_label = iob_to_label(id2label[prediction])
    draw.rectangle(unnormal_box, outline=label2color[predicted_label])
    
    draw.text((unnormal_box[0], unnormal_box[1]), text=predicted_label, fill=label2color[predicted_label], font=font)
    true_label = iob_to_label(id2label[true_id])
    draw.text((unnormal_box[0], unnormal_box[3]), true_label, fill=label2color[true_label], font=font)

original_image